<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Box Plots**


Estimated time needed: **45** minutes


In this lab, you will focus on the visualization of data. The dataset will be provided through an RDBMS, and you will need to use SQL queries to extract the required data.


## Objectives


In this lab you will perform the following:


-   Visualize the distribution of data.

-   Visualize the relationship between two features.

-   Visualize data composition and comparisons using box plots.


### Setup: Connecting to the Database


#### 1. Download the Database File


In [ ]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/QR9YeprUYhOoLafzlLspAw/survey-results-public.sqlite

#### 2. Connect to the Database


**Install the needed libraries**


In [ ]:
!pip install pandas

In [ ]:
!pip install matplotlib

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# Connect to the SQLite database
conn = sqlite3.connect('survey-results-public.sqlite')


## Demo: Basic SQL Queries


#### Demo 1: Count the Number of Rows in the Table


In [ ]:
QUERY = "SELECT COUNT(*) FROM main"
df = pd.read_sql_query(QUERY, conn)
print(df)


#### Demo 2: List All Tables


In [ ]:
QUERY = """
SELECT name as Table_Name 
FROM sqlite_master 
WHERE type = 'table'
"""
pd.read_sql_query(QUERY, conn)


#### Demo 3: Group Data by Age


In [ ]:
QUERY = """
SELECT Age, COUNT(*) as count 
FROM main 
GROUP BY Age 
ORDER BY Age
"""
df_age = pd.read_sql_query(QUERY, conn)
print(df_age)


## Visualizing Data


### Task 1: Visualizing the Distribution of Data


**1. Box Plot of `CompTotal` (Total Compensation)**


Use a box plot to analyze the distribution and outliers in total compensation.


In [ ]:
# ---- Helpers used throughout this lab ----
import numpy as np

# Age is stored as text ranges, so map each range to an approximate number (its midpoint)
AGE_MAP = {
    'Under 18 years old': 17, '18-24 years old': 21, '25-34 years old': 29.5,
    '35-44 years old': 39.5, '45-54 years old': 49.5, '55-64 years old': 59.5,
    '65 years or older': 70,
}
AGE_ORDER = list(AGE_MAP.keys())

# TimeSearching / TimeAnswering are text ranges -> approximate minutes per day
TIME_MAP = {
    'Less than 15 minutes a day': 7.5, '15-30 minutes a day': 22.5,
    '30-60 minutes a day': 45, '60-120 minutes a day': 90, 'Over 120 minutes a day': 150,
}

def years_to_num(series):
    """YearsCodePro is text: turn 'Less than 1 year' / 'More than 50 years' into numbers."""
    return pd.to_numeric(series.replace({'Less than 1 year': '0', 'More than 50 years': '51'}),
                         errors='coerce')

def split_counts(series, top_n=None):
    """Count items in a ';'-separated multi-answer column."""
    counts = series.dropna().str.split(';').explode().str.strip().value_counts()
    return counts.head(top_n) if top_n else counts

def box_by(data, value_col, group_col, order, title, ylabel, log=False):
    """Draw one box per group, in the given order."""
    data = data.dropna(subset=[value_col, group_col])
    groups = [data.loc[data[group_col] == g, value_col] for g in order]
    plt.figure(figsize=(12, 6))
    plt.boxplot(groups)
    plt.xticks(range(1, len(order) + 1), order, rotation=30, ha='right')
    if log:
        plt.yscale('log')   # compensation spans several orders of magnitude
    plt.title(title)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()

# ---- Task 1.1: Box plot of CompTotal ----
comp = pd.read_sql_query("SELECT CompTotal FROM main WHERE CompTotal IS NOT NULL", conn)
comp['CompTotal'] = pd.to_numeric(comp['CompTotal'], errors='coerce')
comp = comp[comp['CompTotal'] > 0]

plt.figure(figsize=(6, 7))
plt.boxplot(comp['CompTotal'])
plt.yscale('log')
plt.title('Box Plot of CompTotal (log scale)')
plt.ylabel('Total compensation (log scale)')
plt.xticks([1], ['CompTotal'])
plt.show()

**2. Box Plot of Age (converted to numeric values)**


Convert the `Age` column into numerical values and visualize the distribution.


In [ ]:
# ---- Task 1.2: Box plot of Age (numeric) ----
age = pd.read_sql_query("SELECT Age FROM main WHERE Age IS NOT NULL", conn)
age['Age_numeric'] = age['Age'].map(AGE_MAP)

plt.figure(figsize=(6, 7))
plt.boxplot(age['Age_numeric'].dropna())
plt.title('Box Plot of Age')
plt.ylabel('Age (approximate, years)')
plt.xticks([1], ['Age'])
plt.show()

### Task 2: Visualizing Relationships in Data


**1. Box Plot of `CompTotal` Grouped by Age Groups:**


Visualize the distribution of compensation across different age groups.


In [ ]:
# ---- Task 2.1: CompTotal by age group ----
ca = pd.read_sql_query("SELECT Age, CompTotal FROM main WHERE CompTotal > 0 AND Age IS NOT NULL", conn)
ca['CompTotal'] = pd.to_numeric(ca['CompTotal'], errors='coerce')
box_by(ca, 'CompTotal', 'Age', [a for a in AGE_ORDER if a in ca['Age'].unique()],
       'CompTotal by Age Group (log scale)', 'Total compensation', log=True)

**2. Box Plot of `CompTotal` Grouped by Job Satisfaction (`JobSatPoints_6`):**


Examine how compensation varies based on job satisfaction levels.


In [ ]:
# ---- Task 2.2: CompTotal by job satisfaction (JobSatPoints_6) ----
cj = pd.read_sql_query("SELECT CompTotal, JobSatPoints_6 FROM main WHERE CompTotal > 0 AND JobSatPoints_6 IS NOT NULL", conn)
cj['CompTotal'] = pd.to_numeric(cj['CompTotal'], errors='coerce')
cj['JobSatPoints_6'] = pd.to_numeric(cj['JobSatPoints_6'], errors='coerce')

# JobSatPoints_6 is a 0-100 score, so group it into satisfaction bands
labels = ['0-20', '21-40', '41-60', '61-80', '81-100']
cj['SatBand'] = pd.cut(cj['JobSatPoints_6'], bins=[-1, 20, 40, 60, 80, 100], labels=labels)
box_by(cj, 'CompTotal', 'SatBand', labels,
       'CompTotal by Job Satisfaction (JobSatPoints_6 bands, log scale)', 'Total compensation', log=True)

### Task 3: Visualizing the Composition of Data


**1. Box Plot of `ConvertedCompYearly` for the Top 5 Developer Types:**


Analyze compensation across the top 5 developer roles.


In [ ]:
# ---- Task 3.1: ConvertedCompYearly for the top 5 developer types ----
dt = pd.read_sql_query("SELECT DevType, ConvertedCompYearly FROM main WHERE ConvertedCompYearly > 0 AND DevType IS NOT NULL", conn)
dt['ConvertedCompYearly'] = pd.to_numeric(dt['ConvertedCompYearly'], errors='coerce')
top5_dev = dt['DevType'].value_counts().head(5).index.tolist()
box_by(dt, 'ConvertedCompYearly', 'DevType', top5_dev,
       'Yearly Compensation for the Top 5 Developer Types (log scale)', 'ConvertedCompYearly (USD)', log=True)

**2. Box Plot of `CompTotal` for the Top 5 Countries:**


Analyze compensation across respondents from the top 5 countries.


In [ ]:
# ---- Task 3.2: CompTotal for the top 5 countries ----
cc = pd.read_sql_query("SELECT Country, CompTotal FROM main WHERE CompTotal > 0 AND Country IS NOT NULL", conn)
cc['CompTotal'] = pd.to_numeric(cc['CompTotal'], errors='coerce')
top5_countries = cc['Country'].value_counts().head(5).index.tolist()
box_by(cc, 'CompTotal', 'Country', top5_countries,
       'CompTotal for the Top 5 Countries (log scale, local currencies)', 'Total compensation', log=True)

### Task 4: Visualizing Comparison of Data


**1. Box Plot of CompTotal Across Employment Types:**


Analyze compensation for different employment types.


In [ ]:
# ---- Task 4.1: CompTotal across employment types ----
ce = pd.read_sql_query("SELECT Employment, CompTotal FROM main WHERE CompTotal > 0 AND Employment IS NOT NULL", conn)
ce['CompTotal'] = pd.to_numeric(ce['CompTotal'], errors='coerce')
top_emp = ce['Employment'].value_counts().head(5).index.tolist()
box_by(ce, 'CompTotal', 'Employment', top_emp,
       'CompTotal by Employment Type (top 5, log scale)', 'Total compensation', log=True)

**2. Box Plot of `YearsCodePro` by Job Satisfaction (`JobSatPoints_6`):**


Examine the distribution of professional coding years by job satisfaction levels.


In [ ]:
# ---- Task 4.2: YearsCodePro by job satisfaction (JobSatPoints_6) ----
yj = pd.read_sql_query("SELECT YearsCodePro, JobSatPoints_6 FROM main WHERE YearsCodePro IS NOT NULL AND JobSatPoints_6 IS NOT NULL", conn)
yj['YearsCodePro_num'] = years_to_num(yj['YearsCodePro'])
yj['JobSatPoints_6'] = pd.to_numeric(yj['JobSatPoints_6'], errors='coerce')
labels = ['0-20', '21-40', '41-60', '61-80', '81-100']
yj['SatBand'] = pd.cut(yj['JobSatPoints_6'], bins=[-1, 20, 40, 60, 80, 100], labels=labels)
box_by(yj, 'YearsCodePro_num', 'SatBand', labels,
       'Years of Professional Coding by Job Satisfaction (JobSatPoints_6 bands)', 'YearsCodePro')

### Final Step: Close the Database Connection


After completing the lab, close the connection to the SQLite database:


In [ ]:
conn.close()

## Summary


In this lab, you used box plots to visualize various aspects of the dataset, focusing on:

- Visualize distributions of compensation and age.

- Explore relationships between compensation, job satisfaction, and professional coding experience.

- Analyze data composition across developer roles and countries.

- Compare compensation across employment types and satisfaction levels.

Box plots provided clear insights into the spread, outliers, and central tendencies of various features in the dataset.


## Authors:
Ayushi Jain


### Other Contributors:
- Rav Ahuja
- Lakshmi Holla
- Malika


<!--## Change Log
|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|               
|2024-10-07|1.2|Madhusudan Moole|Reviewed and updated lab|                                                                                      
|2024-10-06|1.0|Raghul Ramesh|Created lab|-->


Copyright © IBM Corporation. All rights reserved.
